<a href="https://colab.research.google.com/github/marantmir/pos_graduacao_ia_aplicada_sesi_senai_sc/blob/main/aprendizado_profundo/desafio_frota_caminhoes/notebook_v0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Diagnóstico Preditivo de Falhas no Sistema APS em Caminhões com Machine Learning**

##**1. Visão Geral do Projeto**

### **Contexto:**

*   Dataset real da Scania (telemetria de caminhões)
*   Problema de classificação binária (falha APS vs não falha)
*   Alto impacto financeiro e operacional

## **2. CRISP-DM - Estrutura Completa do Projeto**

### &nbsp;&nbsp;&nbsp;&nbsp;**2.1. Business Understanding (Entendimento do Negócio)**

- **Objetivo do negócio:**

  - Reduzir falhas críticas no sistema APS
  - Evitar paradas inesperadas de caminhões

- Regra de ouro:

  - Falso Negativo custa 50x mais que Falso Positivo

  - **-> Função de custo:**

    - FP = 10
    - FN = 500

&nbsp;&nbsp;&nbsp;&nbsp;**! Insight forte:**

&nbsp;&nbsp;&nbsp;&nbsp;O problema NÃO é só prever bem, é minimizar custo operacional

### &nbsp;&nbsp;&nbsp;&nbsp;**2.2. Data Understanding (Entendimento dos Dados)**

- **Dataset:**

  - 60k treino / 16k teste
  - 171 variáveis anonimizadas
  - **Dados**:
    - Sensores
    - Histogramas
    - Contadores

- **Target:**

  - 1 → Falha APS
  - 0 → Sem falha APS

&nbsp;&nbsp;&nbsp;&nbsp;**! Insight:**

&nbsp;&nbsp;&nbsp;&nbsp;Dados complexos e de alta dimensionalidade
Possível desbalanceamento (crítico investigar)

### &nbsp;&nbsp;&nbsp;&nbsp; **2.3 Data Preparation (Preparação dos Dados)**

- **Possíveis etapas:**

  - Tratamento de valores faltantes
  - Normalização / padronização
  - Redução de dimensionalidade (PCA)
  - Balanceamento (SMOTE ou class_weight)
  - Seleção de features

  - Insight:

    - Feature engineering pode ser mais importante que o modelo

### &nbsp;&nbsp;&nbsp;&nbsp; **2.4 Modeling (Modelagem)**

- **Modelos a serem testados:**

  - Deep Learning
  - Random Forest
  - XGBoost
  - Regressão Logística
  - SVM / LightGBM

- **Métricas a serem avaliadas:**

  - Acurácia
  - Precision
  - Recall
  - F1-score
  - ROC-AUC

  - **Ponto chave:**

    - Recall é mais importante que precisão nesse problema

### &nbsp;&nbsp;&nbsp;&nbsp; **2.5 Evaluation (Avaliação)**

- **Critério real de sucesso:**

  - Minimizar custo total

  - **Fórmula:**

    - `Custo = (10 × FP) + (500 × FN)`

  - **Insight:**

    - Melhor modelo ≠ maior acurácia
    - Melhor modelo = menor custo

- **Análises importantes:**

  - Overfitting (treino vs teste)
  - Curva ROC
  - Curva de perda

### &nbsp;&nbsp;&nbsp;&nbsp; **2.6 Deployment (Implantação)**

- **Como aplicar no mundo real:**

  - Monitoramento contínuo da frota
  - Alertas preditivos para manutenção
  - Integração com sistemas de manutenção

- **Evolução do projeto:**

  - Dashboard (Streamlit / Power BI)
  - API de predição
  - Pipeline automatizado (DataOps)

## **3. Insights Estratégicos (Slide de Valor)**

- **Principais aprendizados:**
  - O custo do erro muda completamente a estratégia
  - Recall alto salva dinheiro (evita falhas graves)
  - Modelos mais complexos (XGBoost / DL) tendem a performar melhor
  - Feature engineering é crítico

In [20]:
# =========================================================
# 1. SETUP E INICIALIZAÇÃO DO AMBIENTE
# =========================================================
import sys
import subprocess
from importlib.util import find_spec

def garantir_dependencias(pacotes: list[str] | str) -> None:
    if isinstance(pacotes, str): pacotes = [pacotes]
    print(f"{'='*47}\n      INICIALIZANDO AMBIENTE DE EXECUÇÃO\n{'='*47}")

    for pacote in pacotes:
        pacote_base = pacote.split('==')[0].split('>=')[0].split('<')[0].split('[')[0]
        print(f"Verificando: {pacote_base:<20}", end="")
        if find_spec(pacote_base) is None:
            print(f"| [INSTALANDO]")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", pacote, "--quiet", "--no-cache-dir"])
            except subprocess.CalledProcessError:
                print(f" Erro ao instalar {pacote}")
        else:
            print(f"| [OK]")
    print(f"{'='*47}\nAmbiente pronto para codificação.\n")

garantir_dependencias([
    "tensorflow", "keras", "pandas", "numpy", "scikit-learn",
    "matplotlib", "seaborn", "tqdm", "xgboost"
])

      INICIALIZANDO AMBIENTE DE EXECUÇÃO
Verificando: tensorflow          | [OK]
Verificando: keras               | [OK]
Verificando: pandas              | [OK]
Verificando: numpy               | [OK]
Verificando: scikit-learn        | [INSTALANDO]
Verificando: matplotlib          | [OK]
Verificando: seaborn             | [OK]
Verificando: tqdm                | [OK]
Verificando: xgboost             | [OK]
Ambiente pronto para codificação.



In [23]:
# =========================================================
# 2. IMPORTAÇÕES E DOWNLOAD DO DATASET ÚNICO
# =========================================================
import os
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from pathlib import Path
from typing import Tuple, Optional
import subprocess

# Scikit-Learn e TensorFlow
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks

def preparar_dataset(url: str, nome_arquivo: str) -> None:
    print(f"{'='*87}\nDATASET: {nome_arquivo}\n{'='*87}")
    if not os.path.exists(nome_arquivo) or os.path.getsize(nome_arquivo) < 100:
        print(f"Baixando arquivo...")
        response = requests.get(url, stream=True, timeout=30)
        total_size = int(response.headers.get('content-length', 0))
        with open(nome_arquivo, "wb") as file, tqdm(desc=nome_arquivo, total=total_size, unit='iB', unit_scale=True) as bar:
            for data in response.iter_content(chunk_size=8192):
                bar.update(file.write(data))

    if subprocess.run(["which", "7z"], capture_output=True).returncode != 0:
        subprocess.run(["sudo", "apt-get", "update"], capture_output=True)
        subprocess.run(["sudo", "apt-get", "install", "-y", "p7zip-full"], capture_output=True)

    resultado = subprocess.run(["7z", "x", nome_arquivo, "-y", "-o."], capture_output=True, text=True)
    if resultado.returncode == 0:
        print("Extração concluída!")
    print(f"{'='*87}\n")

def localizar_datasets(diretorio: str = ".") -> Tuple[Optional[Path], Optional[Path]]:
    arquivos_csv = list(Path(diretorio).rglob("*.csv"))

    train_path = next((f for f in arquivos_csv if "train" in f.name.lower()), None)
    test_path = next((f for f in arquivos_csv if "test" in f.name.lower()), None)

    print("-" * 30)
    print(f"Treino localizado: {train_path.name if train_path else 'Não encontrado'}")
    print(f"Teste localizado:  {test_path.name if test_path else 'Não encontrado'}")
    print("-" * 30)

    return train_path, test_path

# URL ÚNICA ORIGINAL DO SEU SCRIPT
URL_GITHUB = "https://github.com/marantmir/pos_graduacao_ia_aplicada_sesi_senai_sc/raw/refs/heads/main/aprendizado_profundo/desafio_frota_caminhoes/data/aps_failure_test_set.7z"
ARQUIVO_LOCAL = "aps_failure_test_set.7z"

preparar_dataset(URL_GITHUB, ARQUIVO_LOCAL)
caminho_treino, caminho_teste = localizar_datasets()

DATASET: aps_failure_test_set.7z
Extração concluída!

------------------------------
Treino localizado: aps_failure_training_set.csv
Teste localizado:  aps_failure_test_set.csv
------------------------------


In [24]:
# =========================================================
# 3. CARREGAMENTO E ANÁLISE EXPLORATÓRIA (EDA)
# =========================================================

# Carregando os datasets localizados pela sua função (tratando 'na' como NaN)
df_train = pd.read_csv(caminho_treino, na_values='na')
df_test = pd.read_csv(caminho_teste, na_values='na')

# Separando Variáveis Explicativas (X) e a Classe Alvo (y)
X_train = df_train.drop("class", axis=1)
y_train = df_train["class"].map({"neg": 0, "pos": 1}).values

X_test = df_test.drop("class", axis=1)
y_test = df_test["class"].map({"neg": 0, "pos": 1}).values

print(f"Dimensões do Treino: {X_train.shape}")
print(f"Dimensões do Teste: {X_test.shape}\n")

# --- ANÁLISE PARA JUSTIFICAR O PIPELINE ---
print("--- Justificativa para o SimpleImputer ---")
nulos_perc = X_train.isna().mean() * 100
col_nulos = nulos_perc[nulos_perc > 0].sort_values(ascending=False)
print(f"{len(col_nulos)} das 170 colunas possuem dados faltantes.")
print(f"A pior coluna tem {col_nulos.max():.2f}% de nulos. Se não usarmos Imputer, perderemos o dataset todo.\n")

print("--- Justificativa para o StandardScaler ---")
# Comparando as distribuições de 5 colunas para ver a diferença de escala
colunas_exemplo = X_train.columns[10:15]
display(X_train[colunas_exemplo].describe().loc[['mean', 'max']])
print("Observação: As escalas variam de unidades a milhões. Modelos matemáticos precisam de padronização.")

Dimensões do Treino: (60000, 170)
Dimensões do Teste: (16000, 170)

--- Justificativa para o SimpleImputer ---
169 das 170 colunas possuem dados faltantes.
A pior coluna tem 82.11% de nulos. Se não usarmos Imputer, perderemos o dataset todo.

--- Justificativa para o StandardScaler ---


,ag_004,ag_005,ag_006,ag_007,ag_008
mean,4.370966e+05,1.108374e+06,1.657818e+06,4.993098e+05,3.556989e+04
max,2.288306e+08,1.791880e+08,9.402067e+07,6.334675e+07,1.770252e+07


Observação: As escalas variam de unidades a milhões. Modelos matemáticos precisam de padronização.


In [27]:
# =========================================================
# 4. PRÉ-PROCESSAMENTO E REGRAS DE NEGÓCIO
# =========================================================

pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

print("Aplicando Pipeline nos dados...")
X_train_scaled = pipeline.fit_transform(X_train)
X_test_scaled = pipeline.transform(X_test)

# Balanço de Classes para focar no custo do falso negativo
neg, pos = np.bincount(y_train)
class_weights = {0: 1.0, 1: (neg / pos)}
print(f"Pesos aplicados: Classe 0 (1.0) | Classe 1 ({class_weights[1]:.2f})")

def cost_function(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return (10 * fp) + (500 * fn)

Aplicando Pipeline nos dados...
Pesos aplicados: Classe 0 (1.0) | Classe 1 (59.00)


In [39]:
# =========================================================
# 5. TREINAMENTO: REDE NEURAL PROFUNDA
# =========================================================

def build_model(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2), # Dropout essencial para evitar overfitting
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ])
    # Trocamos a métrica visual para Recall, que é o foco do negócio
    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.0005),
        loss='binary_crossentropy',
        metrics=[keras.metrics.Recall(name='recall')]
    )
    return model

early_stop = callbacks.EarlyStopping(monitor='val_recall', patience=10, restore_best_weights=True)

print("Treinando Deep Learning... (Isso pode levar alguns minutos)")
dl_model = build_model(X_train_scaled.shape[1])
history = dl_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=0
)
print(f"Treino concluído na época {len(history.history['loss'])} (Graças ao Early Stopping).")

Treinando Deep Learning... (Isso pode levar alguns minutos)
Treino concluído na época 11 (Graças ao Early Stopping).


In [36]:
# =========================================================
# 6. TREINAMENTO: REGRESSÃO, RANDOM FOREST E XGBOOST
# =========================================================

print("Treinando Regressão Logística...")
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

print("Treinando Random Forest...")
rf_model = RandomForestClassifier(class_weight='balanced', n_estimators=100, n_jobs=-1, random_state=42)
rf_model.fit(X_train_scaled, y_train)

print("Treinando XGBoost...")
xgb_model = XGBClassifier(scale_pos_weight=(neg/pos), learning_rate=0.1, n_estimators=100, n_jobs=-1, random_state=42)
xgb_model.fit(X_train_scaled, y_train)

print("Modelos clássicos treinados com sucesso!")

Treinando Regressão Logística...
Treinando Random Forest...
Treinando XGBoost...
Modelos clássicos treinados com sucesso!


In [40]:
# =========================================================
# 7. ANÁLISE EMPÍRICA DE OVERFITTING (TREINO VS TESTE)
# =========================================================
# Um modelo tem overfitting se sua pontuação no Treino for muito
# superior à pontuação no Teste (ele "decorou" mas não generalizou).

print("=== ANÁLISE DE OVERFITTING (RECALL E F1-SCORE) ===")

def analisar_overfitting(nome_modelo, y_treino_pred, y_teste_pred):
    rec_tr = recall_score(y_train, y_treino_pred)
    rec_te = recall_score(y_test, y_teste_pred)
    f1_tr = f1_score(y_train, y_treino_pred)
    f1_te = f1_score(y_test, y_teste_pred)

    print(f"\n[{nome_modelo}]")
    print(f"  Recall -> Treino: {rec_tr:.4f} | Teste: {rec_te:.4f} | Queda: {(rec_tr - rec_te):.4f}")
    print(f"  F1     -> Treino: {f1_tr:.4f} | Teste: {f1_te:.4f} | Queda: {(f1_tr - f1_te):.4f}")

    # Avaliação lógica: Queda maior que 10% indica overfitting
    if (rec_tr - rec_te) > 0.10 or (f1_tr - f1_te) > 0.15:
        print("  ⚠️ ALERTA: Overfitting severo detectado. O modelo decorou os dados de treino.")
    elif (rec_tr - rec_te) < 0:
        print("  ✅ EXCELENTE: Modelo generalizou tão bem que o teste foi melhor que o treino.")
    else:
        print("  ✅ SAUDÁVEL: Modelo reteve a capacidade de generalização.")

# 1. Avaliando Deep Learning (precisamos converter probabilidades em 0 ou 1)
y_pred_tr_dl = (dl_model.predict(X_train_scaled, verbose=0) > 0.5).astype(int).flatten()
y_pred_te_dl = (dl_model.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
analisar_overfitting("Deep Learning (Rede Neural)", y_pred_tr_dl, y_pred_te_dl)

# 2. Avaliando Regressão Logística
analisar_overfitting("Regressão Logística", lr_model.predict(X_train_scaled), lr_model.predict(X_test_scaled))

# 3. Avaliando Random Forest
analisar_overfitting("Random Forest", rf_model.predict(X_train_scaled), rf_model.predict(X_test_scaled))

# 4. Avaliando XGBoost
analisar_overfitting("XGBoost", xgb_model.predict(X_train_scaled), xgb_model.predict(X_test_scaled))

=== ANÁLISE DE OVERFITTING (RECALL E F1-SCORE) ===

[Deep Learning (Rede Neural)]
  Recall -> Treino: 0.9670 | Teste: 0.9787 | Queda: -0.0117
  F1     -> Treino: 0.3153 | Teste: 0.4026 | Queda: -0.0873
  ✅ EXCELENTE: Modelo generalizou tão bem que o teste foi melhor que o treino.

[Regressão Logística]
  Recall -> Treino: 0.9590 | Teste: 0.9227 | Queda: 0.0363
  F1     -> Treino: 0.5644 | Teste: 0.6325 | Queda: -0.0681
  ✅ SAUDÁVEL: Modelo reteve a capacidade de generalização.

[Random Forest]
  Recall -> Treino: 0.9990 | Teste: 0.5653 | Queda: 0.4337
  F1     -> Treino: 0.9995 | Teste: 0.7032 | Queda: 0.2963
  ⚠️ ALERTA: Overfitting severo detectado. O modelo decorou os dados de treino.

[XGBoost]
  Recall -> Treino: 1.0000 | Teste: 0.8880 | Queda: 0.1120
  F1     -> Treino: 0.8799 | Teste: 0.7919 | Queda: 0.0880
  ⚠️ ALERTA: Overfitting severo detectado. O modelo decorou os dados de treino.


In [41]:
# =========================================================
# 8. COMPARAÇÃO FINAL E SELEÇÃO DE MODELO (FOCO NO CUSTO)
# =========================================================

resultados = []

def registrar(nome, y_pred, y_prob):
    resultados.append({
        "Modelo": nome,
        "Acurácia": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Custo Total ($)": cost_function(y_test, y_pred)
    })

registrar("Deep Learning", y_pred_te_dl, dl_model.predict(X_test_scaled, verbose=0).flatten())
registrar("Regressão Logística", lr_model.predict(X_test_scaled), lr_model.predict_proba(X_test_scaled)[:, 1])
registrar("Random Forest", rf_model.predict(X_test_scaled), rf_model.predict_proba(X_test_scaled)[:, 1])
registrar("XGBoost", xgb_model.predict(X_test_scaled), xgb_model.predict_proba(X_test_scaled)[:, 1])

df_resultados = pd.DataFrame(resultados).sort_values(by="Custo Total ($)", ascending=True)

print("\n" + "="*80)
print("COMPARAÇÃO FINAL DOS MODELOS (Ordenado do Menor para o Maior Custo)")
print("="*80)
display(df_resultados.style.format({
    'Acurácia': '{:.4f}', 'Precision': '{:.4f}', 'Recall': '{:.4f}',
    'F1-score': '{:.4f}', 'ROC-AUC': '{:.4f}', 'Custo Total ($)': '${:,.2f}'
}).background_gradient(subset=['Custo Total ($)'], cmap='Greens_r'))


COMPARAÇÃO FINAL DOS MODELOS (Ordenado do Menor para o Maior Custo)


,Modelo,Acurácia,Precision,Recall,F1-score,ROC-AUC,Custo Total ($)
0,Deep Learning,0.9319,0.2535,0.9787,0.4026,0.9884,"$14,810.00"
1,Regressão Logística,0.9749,0.4812,0.9227,0.6325,0.9794,"$18,230.00"
3,XGBoost,0.9891,0.7146,0.8880,0.7919,0.9940,"$22,330.00"
2,Random Forest,0.9888,0.9298,0.5653,0.7032,0.9933,"$81,660.00"
